In [1]:
import pandas as pd
import json
import scanpy as sc
import os


from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

### Convert to h5ad

The data is available only as .rds files native for R. We will convert them to h5ad files using the `scCustomize::as.anndata` function in R.

Run the following script `data_exploration/Perturbseq/supplementary/jiang_2025/convert_to_h5ad.R <input_rds_file> <output_h5ad_file>` for each .rds file to convert it to .h5ad format. 

The converted files should be saved in the `data_exploration/Perturbseq/curation_notebooks/jiang_2025/non_curated/h5ad` directory.

Note that this operation requires R, `scCustomize` and `reticulate` packages to be installed.

The conversion requires up to 160 GB of RAM, depending on the size of the .rds file. 

### Concatenate h5ad files

After converting all .rds files to .h5ad format, we will concatenate them into a single h5ad file for easier downstream analysis.

Uncomment the code in the cell below and run it to concatenate all h5ad files in the `data_exploration/Perturbseq/curation_notebooks/jiang_2025/non_curated/h5ad` directory into a single h5ad file named `data_exploration/Perturbseq/curation_notebooks/jiang_2025/curated/jiang_2025.h5ad`.

In [2]:
# # Collect AnnData objects from the non_curated/h5ad folder
# anndatas = []

# for f in os.listdir('../non_curated/h5ad'):
#     # only process files that start with the study prefix
#     if f.startswith('jiang_2025'):
#         # read the h5ad file into an AnnData object
#         anndata = sc.read_h5ad(f'../non_curated/h5ad/{f}')
#         print(f"Loaded {f}")
#         print(anndata)
        
#         # keep only the desired obs columns to standardize across files
#         anndata.obs = anndata.obs[['sample', 'cell_type', 'sample_ID', 'Batch_info', 'guide', 'gene']]
        
#         # remove the layers attribute if present to avoid compatibility/size issues
#         if anndata.layers:
#             del anndata.layers
#             print('Removed layers slot')
        
#         print("Subset the columns")
#         anndatas.append(anndata)
        
# # Concatenate all collected AnnData objects along observations (cells).
# # - axis='obs' means stacking cells (rows) from each object
# # - join='inner' keeps only variables/columns (genes) common to all objects
# adata_combined = sc.concat(
#     anndatas, 
#     axis='obs', 
#     join="inner"
# )

# # Write the combined AnnData to disk as a single h5ad file
# adata_combined.write_h5ad("../non_curated/h5ad/jiang_2025.h5ad")


# Initialise the dataset object

In [3]:
noncurated_path = '../non_curated/h5ad/jiang_2025.h5ad'
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/jiang_2025.h5ad


/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [4]:
cur_data.adata.obs

,sample,cell_type,sample_ID,Batch_info,guide,gene,batch
82_33_30_1_1_1_1_1_1_1_1_1,A549_TGFB1,A549,sample_1,Rep1,RUNX2g2,RUNX2,0
84_37_68_1_1_1_1_1_1_1_1_1,A549_TGFB1,A549,sample_1,Rep1,RPS6KB1g1,RPS6KB1,0
82_06_13_1_1_1_1_1_1_1_1_1,A549_TGFB1,A549,sample_1,Rep1,RPS6KB1g3,RPS6KB1,0
84_52_73_1_1_1_1_1_1_1_1_1,A549_TGFB1,A549,sample_1,Rep1,PPP2CAg2,PPP2CA,0
83_37_69_1_1_1_1_1_1_1_1_1,A549_TGFB1,A549,sample_1,Rep1,RUNX1g2,RUNX1,0
...,...,...,...,...,...,...,...
82_83_54_2_2,MCF7_INS,MCF7,sample_16,Rep2,PIK3CAg3,PIK3CA,4
81_04_45_2_2,MCF7_INS,MCF7,sample_16,Rep2,TSC2g3,TSC2,4
82_92_06_2_2,MCF7_INS,MCF7,sample_16,Rep2,FOXO3g2,FOXO3,4
84_83_92_2_2,MCF7_INS,MCF7,sample_16,Rep2,XBP1g1,XBP1,4


In [5]:
cur_data.adata.var

""
TSPAN6
TNMD
DPM1
SCYL3
C1orf112
...
AC084033.2
NUTF2P2
AC068620.2
AC006023.1


### Filter out cells with unknown treatment

In [6]:
print(f"Number of cells with unknown treatment: {cur_data.adata.obs.query('sample == \"unknown_unknown\"').shape[0]}")

print("Filtering out cells with unknown treatment...")
cur_data.adata = cur_data.adata[cur_data.adata.obs['sample'] != 'unknown_unknown'].copy()
print(f"Number of cells after filtering: {cur_data.adata.shape[0]}")

Number of cells with unknown treatment: 10128
Filtering out cells with unknown treatment...
Number of cells after filtering: 1618348


/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1791: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


### Add `treatment` column derived from `sample` column

In [7]:
cur_data.adata.obs['treatment'] = cur_data.adata.obs['sample'].str.split('_').str[1]

### Add guide RNA information

Note that NT - non-targeting control guides are missing. The authors have been contacted to clarify whether these were included in the original data and if so, to provide the guide sequences for these guides.

In [8]:
# download the guide RNA spreadsheet
download_file(
    url="https://static-content.springer.com/esm/art%3A10.1038%2Fs41556-025-01622-z/MediaObjects/41556_2025_1622_MOESM3_ESM.xlsx",
    dest_path="../supplementary/jiang_2025/jiang_2025_guide_info.xlsx"
)

# read in the guide RNA spreadsheet
# guides for the K562 essential day 6 library are in "TabB_K562_day6_library"
guide_info_dict = pd.read_excel("../supplementary/jiang_2025/jiang_2025_guide_info.xlsx", 
                              header=2,
                              sheet_name=['ST1a_IFNG_gRNAs', 'ST1b_IFNB_gRNAs', 'ST1c_TGFB_gRNAs', 'ST1d_TNFA_gRNAs', 'ST1e_INS_gRNAs'])#, sheet_name="TabC_RPE1_day7_library")
# clean up the keys to keep only the treatment name (IFNG, IFNB, TGFB, TNFA, INS)
guide_info_dict = {k.split('_')[1]: v for k,v in guide_info_dict.items()}
# concatenate the guide info dataframes into a single dataframe with a new column for treatment
guide_info_df = (
    pd.concat(guide_info_dict)
    .reset_index(drop=True)
    .rename(columns={'gRNA Name': 'guide', 'gRNA Sequence': 'guide_sequence'})
    [['guide', 'guide_sequence']]
    .drop_duplicates()
)

# read non-targeting guides - these are missing from the main guide info sheets, but were provided at the request by the authors
non_targeting_guides_df = pd.read_table("../supplementary/jiang_2025/jiang_2025_nt_guides.txt", header=None, names=['guide_sequence'])
# reshape
non_targeting_guides_df = pd.concat([non_targeting_guides_df[non_targeting_guides_df['guide_sequence'].str.startswith('>NTg')].reset_index(drop=True), 
           non_targeting_guides_df[~non_targeting_guides_df['guide_sequence'].str.startswith('>NTg')].reset_index(drop=True)],
          axis=1,
          ignore_index=True).rename(columns={0: 'guide', 1: 'guide_sequence'})

non_targeting_guides_df['guide'] = non_targeting_guides_df['guide'].str.replace('>', '').str.strip()
# remove the common prefix from the guide sequences
non_targeting_guides_df['guide_sequence'] = non_targeting_guides_df['guide_sequence'].str.replace('GTGGAAAGGACGAAACACCG','')
# keep first 20 bases of the guide sequence to match the format in the main guide info sheet
non_targeting_guides_df['guide_sequence'] = non_targeting_guides_df['guide_sequence'].str[:20]

# concatenate the non-targeting guides with the main guide info dataframe
guide_info_df = pd.concat([guide_info_df, non_targeting_guides_df], ignore_index=True)

guide_info_df

File ../supplementary/jiang_2025/jiang_2025_guide_info.xlsx already exists. Skipping download.


,guide,guide_sequence
0,ATF3g1,GCGGGCTGAAGGGTGCGCTC
1,ATF3g2,GGCTGAAGGGTGCGCTCGGG
2,ATF3g3,TGAAGGGTGCGCTCGGGCGG
3,ATF5g1,AGGCTACAGAGCCATGGCCG
4,ATF5g2,GGGTCGAGGCTACAGAGCCA
...,...,...
663,NTg10,GTAAATTAATGTAACTACCG
664,NTg11,CCATTCTCAACCGGTCCAAT
665,NTg12,ACCCATGAGTTAAGTTTTCT
666,NTg13,CGGCACACCAATGCGTTCGT


In [9]:
# merge the guide info with the obs dataframe to add guide sequences
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='guide', how='left')

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Rename relevant metadata columns

`Sample_ID` is 16 separate libraries (from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM8609800) -> `technical_replicate`

`Batch_info` is two separate sequencing events for each condition -> `biological_replicate`

`cell_type` is acually cell line -> `cell_line`

`guide` -> `perturbation_name`

In [10]:
cur_data.adata.obs = cur_data.adata.obs.rename(columns={'sample_ID': 'technical_replicate', 
                                                        'Batch_info': 'biological_replicate',
                                                        'cell_type': 'cell_line',
                                                        'guide': 'perturbation_name'})

### Standardise perturbation targets

In [11]:
# replace non targeting guides with "NTg" in the perturbation name to "control_nontargeting"
cur_data.adata.obs['gene'] = cur_data.adata.obs['gene'].replace('NT', 'control_nontargeting')

/tmp/ipykernel_1182978/2300470366.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_data.adata.obs['gene'] = cur_data.adata.obs['gene'].replace('NT', 'control_nontargeting')


In [12]:
cur_data.standardize_genes(
    slot='obs',
    input_column='gene',
    input_column_type='gene_symbol',
    multiple_entries=False
)

Mapping gene symbols: 100%|████████████████████████████████████| 219/219 [00:00<00:00, 23165.35it/s]


--------------------------------------------------
Successfully mapped 219 out of 219 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: []
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Add `perturbed_target_number` column

In [13]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

Counted entries in column perturbed_target_symbol of adata.obs and stored in perturbed_target_number


### Encode chromosomes as integers

In [14]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


In [15]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_chromosome_encoding'])

Observation data:
DataFrame shape: (1618348, 2)
--------------------------------------------------
        perturbation_name  perturbed_target_chromosome_encoding
index                                                          
0                 RUNX2g2                                     6
1               RPS6KB1g1                                    17
2               RPS6KB1g3                                    17
3                PPP2CAg2                                     5
4                 RUNX1g2                                    21
...                   ...                                   ...
1618343          PIK3CAg3                                     3
1618344            TSC2g3                                    16
1618345           FOXO3g2                                     6
1618346            XBP1g1                                    22
1618347          PIK3CAg1                                     3

[1618348 rows x 2 columns]
-----------------------------------------

### Curate cell line information

In [16]:
cur_data.standardize_ontology(
    input_column='cell_line',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

Mapped 4 cell_line ontology terms from `cell_line` column to ontology terms
DataFrame shape: (4, 4)
--------------------------------------------------
  input_column input_column_lower name_lower  ontology_id
0         A549               a549       a549  CLO:0001601
2         HT29               ht29       ht29  CLO:0004283
3         K562               k562       k562  CLO:0007050
4         MCF7               mcf7       mcf7  CLO:0007606
--------------------------------------------------
2 cell_line ontology terms from `cell_line` column could not be mapped to ontology terms
DataFrame shape: (2, 4)
--------------------------------------------------
  input_column input_column_lower name_lower ontology_id
1         HAP1               hap1        NaN         NaN
5        BXPC3              bxpc3        NaN         NaN
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [17]:
# manually map HAP1 and BXPC3
cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line_label'] = cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line'].replace({
    'HAP1': 'HAP-1',
    'BXPC3': 'BxPC-3 cell'
})
cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line_id'] = cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line'].replace({
    'HAP1': 'EFO:0007598',
    'BXPC3': 'CLO:0002065'
})

/tmp/ipykernel_1182978/3980960887.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line_label'] = cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line'].replace({
/tmp/ipykernel_1182978/3980960887.py:6: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line_id'] = cur_data.adata.obs.loc[cur_data.adata.obs['cell_line'].isin(['HAP1', 'BXPC3']), 'cell_line'].rep

### Curate cell type information

In [18]:
cur_data.adata.obs['cell_type'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'myeloid cell',
    'K562': 'lymphoblast',
    'A549': 'pulmonary alveolar type 2 cell',
    'HT29': 'enterocyte',
    'BXPC3': None,
    'MCF7': 'luminal epithelial cell of mammary gland'
})

In [19]:
cur_data.standardize_ontology(
    input_column='cell_type',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 5 cell_type ontology terms from `cell_type` column to ontology terms
DataFrame shape: (5, 4)
--------------------------------------------------
                               input_column  \
0            pulmonary alveolar type 2 cell   
1                              myeloid cell   
2                                enterocyte   
3                               lymphoblast   
4  luminal epithelial cell of mammary gland   

                         input_column_lower  \
0            pulmonary alveolar type 2 cell   
1                              myeloid cell   
2                                enterocyte   
3                               lymphoblast   
4  luminal epithelial cell of mammary gland   

                                 name_lower ontology_id  
0            pulmonary alveolar type 2 cell  CL:0002063  
1                              myeloid cell  CL:0000763  
2                                enterocyte  CL:0000584  
3                               lymphoblast  CL:001

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate tissue information


In [20]:
cur_data.adata.obs['tissue'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'bone marrow',
    'K562': 'blood',
    'A549': 'lung',
    'HT29': 'colon',
    'BXPC3': 'pancreas',
    'MCF7': 'breast'
})

In [21]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 6 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (6, 4)
--------------------------------------------------
  input_column input_column_lower   name_lower     ontology_id
0         lung               lung         lung  UBERON:0002048
1  bone marrow        bone marrow  bone marrow  UBERON:0002371
2        colon              colon        colon  UBERON:0001155
3        blood              blood        blood  UBERON:0000178
4       breast             breast       breast  UBERON:0000310
5     pancreas           pancreas     pancreas  UBERON:0001264
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate disease information

In [22]:
cur_data.adata.obs['disease'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'leukemia, myeloid, accelerated-phase',
    'K562': 'chronic myelogenous leukemia, BCR-ABL1 positive',
    'A549': 'lung adenocarcinoma',
    'HT29': 'colon adenocarcinoma',
    'BXPC3': 'pancreatic adenocarcinoma',
    'MCF7': 'invasive breast carcinoma'
})

In [23]:
cur_data.standardize_ontology(
    input_column='disease',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

Mapped 6 disease ontology terms from `disease` column to ontology terms
DataFrame shape: (6, 4)
--------------------------------------------------
                                      input_column  \
0                              lung adenocarcinoma   
1             leukemia, myeloid, accelerated-phase   
2                             colon adenocarcinoma   
3  chronic myelogenous leukemia, BCR-ABL1 positive   
4                        invasive breast carcinoma   
5                        pancreatic adenocarcinoma   

                                input_column_lower  \
0                              lung adenocarcinoma   
1             leukemia, myeloid, accelerated-phase   
2                             colon adenocarcinoma   
3  chronic myelogenous leukemia, bcr-abl1 positive   
4                        invasive breast carcinoma   
5                        pancreatic adenocarcinoma   

                                        name_lower    ontology_id  
0                          

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate sex information

In [24]:
cur_data.adata.obs['sex_label'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'male',
    'K562': 'female',
    'A549': 'male',
    'HT29': 'female',
    'BXPC3': 'female',
    'MCF7': 'female'
})

### Curate developmental stage information

In [25]:
cur_data.adata.obs['developmental_stage_label'] = cur_data.adata.obs['cell_line'].map({
    'HAP1': 'adult',
    'K562': 'adult',
    'A549': 'adult',
    'HT29': 'adult',
    'BXPC3': 'senior adult',
    'MCF7': 'senior adult'
})

### Curate treatment information

In [26]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['treatment'].map({
    'TGFB1': 'TGFB1',
    'INS': 'insulin hormone',
    'TNFA': 'TNFA',
    'IFNB': 'IFNB',
    'IFNG': 'IFNG'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['treatment'].map({
    'TGFB1': 'PR:P01137',
    'INS': 'PR:000050339',
    'TNFA': 'PR:P01375',
    'IFNB': 'PR:000008924',
    'IFNG': 'PR:P01579'
})

In [27]:
cur_data.adata.obs

,cell_line,treatment,perturbation_name,guide_sequence,gene,technical_replicate,batch,biological_replicate,sample,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding,cell_line_label,cell_line_id,cell_type,cell_type_label,cell_type_id,tissue,tissue_label,tissue_id,disease,disease_label,disease_id,sex_label,developmental_stage_label,treatment_label,treatment_id
0,A549,TGFB1,RUNX2g2,CGGTGCAAACTTTCTCCAGG,RUNX2,sample_1,0,Rep1,A549_TGFB1,ENSG00000124813,RUNX2,protein_coding,chr6:45328157-45664349;1,6,0,1,6,A549 cell,CLO:0001601,pulmonary alveolar type 2 cell,pulmonary alveolar type 2 cell,CL:0002063,lung,lung,UBERON:0002048,lung adenocarcinoma,lung adenocarcinoma,MONDO:0005061,male,adult,TGFB1,PR:P01137
1,A549,TGFB1,RPS6KB1g1,CCGGGCCCATGAGGCGACGA,RPS6KB1,sample_1,0,Rep1,A549_TGFB1,ENSG00000108443,RPS6KB1,protein_coding,chr17:59893046-59950574;1,17,1,1,17,A549 cell,CLO:0001601,pulmonary alveolar type 2 cell,pulmonary alveolar type 2 cell,CL:0002063,lung,lung,UBERON:0002048,lung adenocarcinoma,lung adenocarcinoma,MONDO:0005061,male,adult,TGFB1,PR:P01137
2,A549,TGFB1,RPS6KB1g3,GCGGCGGGTCCGGGCCCATG,RPS6KB1,sample_1,0,Rep1,A549_TGFB1,ENSG00000108443,RPS6KB1,protein_coding,chr17:59893046-59950574;1,17,2,1,17,A549 cell,CLO:0001601,pulmonary alveolar type 2 cell,pulmonary alveolar type 2 cell,CL:0002063,lung,lung,UBERON:0002048,lung adenocarcinoma,lung adenocarcinoma,MONDO:0005061,male,adult,TGFB1,PR:P01137
3,A549,TGFB1,PPP2CAg2,GCCTCCTCCTCCGCTCGCTG,PPP2CA,sample_1,0,Rep1,A549_TGFB1,ENSG00000113575,PPP2CA,protein_coding,chr5:134193978-134226083;-1,5,3,1,5,A549 cell,CLO:0001601,pulmonary alveolar type 2 cell,pulmonary alveolar type 2 cell,CL:0002063,lung,lung,UBERON:0002048,lung adenocarcinoma,lung adenocarcinoma,MONDO:0005061,male,adult,TGFB1,PR:P01137
4,A549,TGFB1,RUNX1g2,CGGCGCAGGGCCGGGCAGCG,RUNX1,sample_1,0,Rep1,A549_TGFB1,ENSG00000159216,RUNX1,protein_coding,chr21:34787801-36004667;-1,21,4,1,21,A549 cell,CLO:0001601,pulmonary alveolar type 2 cell,pulmonary alveolar type 2 cell,CL:0002063,lung,lung,UBERON:0002048,lung adenocarcinoma,lung adenocarcinoma,MONDO:0005061,male,adult,TGFB1,PR:P01137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1618343,MCF7,INS,PIK3CAg3,GTGTCGGGCTGCTGCTGCCG,PIK3CA,sample_16,4,Rep2,MCF7_INS,ENSG00000121879,PIK3CA,protein_coding,chr3:179148114-179240093;1,3,1618343,1,3,MCF7 cell,CLO:0007606,luminal epithelial cell of mammary gland,luminal epithelial cell of mammary gland,CL:0002326,breast,breast,UBERON:0000310,invasive breast carcinoma,invasive breast carcinoma,MONDO:0006256,female,senior adult,insulin hormone,PR:000050339
1618344,MCF7,INS,TSC2g3,GTGCGCCTTTCTCCGCGTCG,TSC2,sample_16,4,Rep2,MCF7_INS,ENSG00000103197,TSC2,protein_coding,chr16:2047345-2089491;1,16,1618344,1,16,MCF7 cell,CLO:0007606,luminal epithelial cell of mammary gland,luminal epithelial cell of mammary gland,CL:0002326,breast,breast,UBERON:0000310,invasive breast carcinoma,invasive breast carcinoma,MONDO:0006256,female,senior adult,insulin hormone,PR:000050339
1618345,MCF7,INS,FOXO3g2,CCTCCCTTCCACGAGCAGGG,FOXO3,sample_16,4,Rep2,MCF7_INS,ENSG00000118689,FOXO3,protein_coding,chr6:108559652-108684774;1,6,1618345,1,6,MCF7 cell,CLO:0007606,luminal epithelial cell of mammary gland,luminal epithelial cell of mammary gland,CL:0002326,breast,breast,UBERON:0000310,invasive breast carcinoma,invasive breast carcinoma,MONDO:0006256,female,senior adult,insulin hormone,PR:000050339
1618346,MCF7,INS,XBP1g1,CAGAACTTTAGGGGTCCCGT,XBP1,sample_16,4,Rep2,MCF7_INS,ENSG00000100219,XBP1,protein_coding,chr22:28794555-28800597;-1,22,1618346,1,22,MCF7 cell,CLO:0007606,luminal epithelial cell of mammary gland,luminal epithelial cell of mammary gland,CL:0002326,breast,breast,UBERON:0000310,invasive breast carcinoma,invasive breast carcinoma,MONDO:00

### Add metadata

In [28]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        # "treatment_label": None,
        # "treatment_id": None,
        #----- replicate -----#
        # "technical_replicate": None,
        # "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "cell_line",
        "model_system_id": None,
        #----- tissue -----#
        # "tissue": "blood",
        #----- cell line -----#
        # "cell_line_label": "K 562 cell",
        # "cell_line_id": None,
        #----- cell type -----#
        # "cell_type_label": "lymphoblast",
        # "cell_type_id": None,
        #----- disease -----#
        # "disease_label": "chronic myelogenous leukemia, BCR-ABL1 positive",
        # "disease_id": None,
        #----- timepoint -----#
        "timepoint": "P13DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        # "sex_label": "female",
        "sex_id": None,
        #----- developmental stage -----#
        # "developmental_stage_label": "adult",
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "Systematic reconstruction of molecular pathway signatures using scalable single-cell perturbation screens",
        "study_uri": "https://doi.org/10.1038/s41556-025-01622-z",
        "study_year": 2025,
        #----- authors -----#
        "first_author": "Longda Jiang",
        "last_author": "Rahul Satija",
        #----- experiment metadata -----#
        "experiment_title": "CRISPRi Perturb-seq of IFNb, IFNg, TNFa, TGFb and insulin pathway regulators in IFNb/IFNg/TNFa/TGFb/insulin-stimulated A549, MCF-7, HT-29, HAP1, K562 and BXPC3 cells.",
        "experiment_summary": """
        The goal of the study was to build a database of molecular response signatures and examine their changes across various cellular contexts. A pooled CRISPRi Perturb-seq screen was performed to characterize the molecular responses to perturbations of known IFNb, IFNg, TNFa, TGFb and insulin pathway regulators. To this end, A549, MCF-7, HT-29, HAP1, K562 and BXPC3 cells were transfected with Doench pathway-sepcific sub-libraries and stimulated with IFNb/IFNg/TNFa/TGFb/insulin for 24h. At this point, the cells were harvested, fixed and frozen for batch processing and sequencing using Ultima Genomics UG100 platform.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-MeCP2",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "Human CRISPR Inhibition Pooled Library (Dolcetto)",
        "library_uri": "https://www.addgene.org/pooled-library/broadgpp-human-crispri-dolcetto/",
        "library_manufacturer": "Doench lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "3",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "Parse Biosciences Evercode Whole Transcriptome Mega v1 kit",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Ultima Genomics UG100",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "Seurat",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        #----- license -----#
        "license_label": "free to use license",
        "license_id": "SWO:1000061",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE281048",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE281048",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE281048_Seurat_object_*_Perturb_seq.rds.gz",
            },
            {
                "dataset_accession": "Seurat_object_*_Perturb_seq.rds",
                "dataset_uri": "https://doi.org/10.5281/zenodo.14518762",
                "dataset_description": "Seurat objects of the Perturb-seq data in .rds format",
                "dataset_file_name": "Seurat_object_*_Perturb_seq.rds",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_id added to adata.obs
Column study_title added to adata.obs
Column study_uri added to adata.obs
Column study_year added to adata.obs
Column first_author added to adata.obs
Column last_author added to adata.obs
Column experiment_title added to adata.obs
Column experiment_summary added to adata.obs
Column number_of_perturbed_targets added to adata.obs
Column number_of_perturbed_samples added to adata.obs
Column library_generation_type_id 

### Match schema column order

In [33]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [30]:
cur_data.validate_data(slot='obs', verbose=True)

2026-03-02 13:43:08,408 INFO curation_tools.curation_tools: adata.obs is valid according to the obs_schema.


,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint,treatment_label,treatment_id,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,cell_line_label,cell_line_id,sex_label,sex_id,developmental_stage_label,developmental_stage_id,disease_label,disease_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,jiang_2025,1,Perturb-seq,<NA>,<NA>,RUNX2g2,chr6:45328157-45664349;1,6,6,1,ENSG00000124813,RUNX2,protein_coding,CGGTGCAAACTTTCTCCAGG,CRISPRi,<NA>,P13DT0H0M0S,TGFB1,PR:P01137,sample_1,Rep1,cell_line,<NA>,Homo sapiens,lung,UBERON:0002048,pulmonary alveolar type 2 cell,CL:0002063,A549 cell,CLO:0001601,male,<NA>,adult,<NA>,lung adenocarcinoma,MONDO:0005061,Systematic reconstruction of molecular pathway...,https://doi.org/10.1038/s41556-025-01622-z,2025,Longda Jiang,Rahul Satija,"CRISPRi Perturb-seq of IFNb, IFNg, TNFa, TGFb ...",\n The goal of the study was to build a...,219,1618348,EFO:0022868,endogenous,<NA>,dCas9-KRAB-MeCP2,<NA>,lentivirus transduction,<NA>,lentivirus transduction,<NA>,random locus integration,<NA>,random locus integration,<NA>,constitutive transgene expression,<NA>,constitutive transgene expression,Human CRISPR Inhibition Pooled Library (Dolcetto),https://www.addgene.org/pooled-library/broadgp...,<NA>,pooled,<NA>,focused,<NA>,inhibition,Doench lab,3,3,668,<NA>,<NA>,high-dimensional assay,<NA>,transcriptomic,<NA>,single-cell rna-seq,<NA>,Perturb-seq,<NA>,<NA>,Parse Biosciences Evercode Whole Transcriptome...,<NA>,Ultima Genomics UG100,<NA>,barcode sequencing,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE281048"", ""dataset_u...",free to use license,SWO:1000061
1,jiang_2025,2,Perturb-seq,<NA>,<NA>,RPS6KB1g1,chr17:59893046-59950574;1,17,17,1,ENSG00000108443,RPS6KB1,protein_coding,CCGGGCCCATGAGGCGACGA,CRISPRi,<NA>,P13DT0H0M0S,TGFB1,PR:P01137,sample_1,Rep1,cell_line,<NA>,Homo sapiens,lung,UBERON:0002048,pulmonary alveolar type 2 cell,CL:0002063,A549 cell,CLO:0001601,male,<NA>,adult,<NA>,lung adenocarcinoma,MONDO:0005061,Systematic reconstruction of molecular pathway...,https://doi.org/10.1038/s41556-025-01622-z,2025,Longda Jiang,Rahul Satija,"CRISPRi Perturb-seq of IFNb, IFNg, TNFa, TGFb ...",\n The goal of the study was to build a...,219,1618348,EFO:0022868,endoge

In [36]:
cur_data.adata.obs

,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint,treatment_label,treatment_id,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,cell_line_label,cell_line_id,sex_label,sex_id,developmental_stage_label,developmental_stage_id,disease_label,disease_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,jiang_2025,1,Perturb-seq,<NA>,<NA>,RUNX2g2,chr6:45328157-45664349;1,6,6,1,ENSG00000124813,RUNX2,protein_coding,CGGTGCAAACTTTCTCCAGG,CRISPRi,<NA>,P13DT0H0M0S,TGFB1,PR:P01137,sample_1,Rep1,cell_line,<NA>,Homo sapiens,lung,UBERON:0002048,pulmonary alveolar type 2 cell,CL:0002063,A549 cell,CLO:0001601,male,<NA>,adult,<NA>,lung adenocarcinoma,MONDO:0005061,Systematic reconstruction of molecular pathway...,https://doi.org/10.1038/s41556-025-01622-z,2025,Longda Jiang,Rahul Satija,"CRISPRi Perturb-seq of IFNb, IFNg, TNFa, TGFb ...",\n The goal of the study was to build a...,219,1618348,EFO:0022868,endogenous,<NA>,dCas9-KRAB-MeCP2,<NA>,lentivirus transduction,<NA>,lentivirus transduction,<NA>,random locus integration,<NA>,random locus integration,<NA>,constitutive transgene expression,<NA>,constitutive transgene expression,Human CRISPR Inhibition Pooled Library (Dolcetto),https://www.addgene.org/pooled-library/broadgp...,<NA>,pooled,<NA>,focused,<NA>,inhibition,Doench lab,3,3,668,<NA>,<NA>,high-dimensional assay,<NA>,transcriptomic,<NA>,single-cell rna-seq,<NA>,Perturb-seq,<NA>,<NA>,Parse Biosciences Evercode Whole Transcriptome...,<NA>,Ultima Genomics UG100,<NA>,barcode sequencing,<NA>,CellRanger,<NA>,Seurat,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE281048"", ""dataset_u...",free to use license,SWO:1000061
1,jiang_2025,2,Perturb-seq,<NA>,<NA>,RPS6KB1g1,chr17:59893046-59950574;1,17,17,1,ENSG00000108443,RPS6KB1,protein_coding,CCGGGCCCATGAGGCGACGA,CRISPRi,<NA>,P13DT0H0M0S,TGFB1,PR:P01137,sample_1,Rep1,cell_line,<NA>,Homo sapiens,lung,UBERON:0002048,pulmonary alveolar type 2 cell,CL:0002063,A549 cell,CLO:0001601,male,<NA>,adult,<NA>,lung adenocarcinoma,MONDO:0005061,Systematic reconstruction of molecular pathway...,https://doi.org/10.1038/s41556-025-01622-z,2025,Longda Jiang,Rahul Satija,"CRISPRi Perturb-seq of IFNb, IFNg, TNFa, TGFb ...",\n The goal of the study was to build a...,219,1618348,EFO:0022868,endoge

In [37]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_symbol', 'perturbed_target_ensg', 'perturbed_target_coord'])

Observation data:
DataFrame shape: (1618348, 4)
--------------------------------------------------
        perturbation_name perturbed_target_symbol perturbed_target_ensg  \
0                 RUNX2g2                   RUNX2       ENSG00000124813   
1               RPS6KB1g1                 RPS6KB1       ENSG00000108443   
2               RPS6KB1g3                 RPS6KB1       ENSG00000108443   
3                PPP2CAg2                  PPP2CA       ENSG00000113575   
4                 RUNX1g2                   RUNX1       ENSG00000159216   
...                   ...                     ...                   ...   
1618343          PIK3CAg3                  PIK3CA       ENSG00000121879   
1618344            TSC2g3                    TSC2       ENSG00000103197   
1618345           FOXO3g2                   FOXO3       ENSG00000118689   
1618346            XBP1g1                    XBP1       ENSG00000100219   
1618347          PIK3CAg1                  PIK3CA       ENSG00000121879   



# VAR slot curation

### Standardise genes

In [54]:
cur_data.adata.var

,gene_name,ensembl_gene_id,gene_symbol,original_index
index,,,,
0,TSPAN6,ENSG00000000003,TSPAN6,0
1,TNMD,ENSG00000000005,TNMD,1
2,DPM1,ENSG00000000419,DPM1,2
3,SCYL3,ENSG00000000457,SCYL3,3
4,C1orf112,ENSG00000000460,FIRRM,4
...,...,...,...,...
33051,AC084033.2,ENSG00000166896,ATP23,33051
33052,NUTF2P2,ENSG00000258300,NUTF2P2,33052
33053,AC068620.2,ENSG00000109265,CRACD,33053


In [ ]:
cur_data.adata.var['gene_name'] = cur_data.adata.var.index

In [53]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_name",
    input_column_type="gene_symbol",
    remove_version=True,
    multiple_entries=False
)

Removed version numbers from gene_name


Mapping gene symbols: 100%|█████████████████████████████████| 29973/29973 [00:05<00:00, 5050.15it/s]


--------------------------------------------------
Successfully mapped 28239 out of 29973 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: ['AC009533', 'AC087235', 'AC116050', 'AC062028', 'LINC00266-1', 'TMEM155', 'AC090152', 'PRR25', 'AC106795', 'AL512625', 'AC098864', 'C9orf106', 'AC093909', 'FAM153B', 'AMYH02020865', 'AC138393', 'AC026412', 'AC069277', 'AC146944', 'AL672167', 'AL359955', 'AC131392', 'RF00019', 'RF00096', 'RF00012', 'RF00100', 'RF00091', 'AL137845', 'LINC01317', 'AL353740', 'AL133216', 'AC144450', 'AL162293', 'FLJ45513', 'AL133464', 'AC073464', 'AC083899', 'AL163540', 'AC021218', 'AC007389', 'AC136759', 'AL356414', 'AP001043', 'FAM239C', 'FAM239B', 'BX890604', 'AC019155', 'Z99774', 'AC006305', 'AC117402', 'RF00432', 'Z97192', 'SPHAR', 'AL590867', 'AC093724', 'AL049873', 'AL731661', 'AC005562', 'AL391419', 'AC092821', 'AP002358', 'AC090921', 'AP000317', 'BX322639', 'AC138409', 'AL592293', 'AC127502', 'AC245297', 'AC092506', '

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Validate var metadata

In [55]:
cur_data.validate_data(slot='var')

2026-03-02 15:04:50,226 INFO curation_tools.curation_tools: adata.var is valid according to the var_schema.


,ensembl_gene_id,gene_symbol
index,,
0,ENSG00000000003,TSPAN6
1,ENSG00000000005,TNMD
2,ENSG00000000419,DPM1
3,ENSG00000000457,SCYL3
4,ENSG00000000460,FIRRM
...,...,...
33051,ENSG00000166896,ATP23
33052,ENSG00000258300,NUTF2P2
33053,ENSG00000109265,CRACD


# Save the dataset

In [57]:
cur_data.save_curated_data_h5ad()

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)
... storing 'dataset_id' as categorical
... storing 'data_modality' as categorical
... storing 'significance_criteria' as categorical
... storing 'perturbation_name' as categorical
... storing 'perturbed_target_coord' as categorical
... storing 'perturbed_target_chromosome' as categorical
... storing 'perturbed_target_ensg' as categorical
... storing 'perturbed_target_symbol' as categorical
... storing 'perturbed_target_biotype' as categorical
... storing 'guide_sequence' as categorical
... storing 'perturbation_type_label' as categorical
... storing 'perturbation

✅ Curated h5ad data saved to ../curated/h5ad/jiang_2025_curated.h5ad


In [58]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/jiang_2025_curated_metadata.parquet


# Upload to BigQuery

In [59]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/jiang_2025_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file ../curated/parquet/jiang_2025_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 1618348 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [61]:
!gcloud storage cp ../curated/h5ad/jiang_2025_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../curated/h5ad/jiang_2025_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/jiang_2025_curated.h5ad
  Completed files 32/1 | 68.6GiB/68.6GiB | 348.2MiB/s                          

Average throughput: 321.5MiB/s


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud